# Lab 1: One qubit

**Goals:** build single-qubit circuits, see superposition on the Bloch
sphere, and connect amplitudes to measurement statistics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/qc-workshop/blob/main/solutions/01_one_qubit.ipynb)

**Solution notebook.**

In [ ]:
# Install pinned versions (takes about a minute in Colab; already done locally)
%pip install -q qiskit==2.6.1 qiskit-aer==0.18.2 qiskit-ibm-runtime==0.51.0 matplotlib pylatexenc

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def run(qc, shots=1000, backend=sim):
    """Run a circuit and return the measurement counts."""
    return backend.run(transpile(qc, backend), shots=shots).result().get_counts()

from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_bloch_multivector
import numpy as np

## 1.1 The state of a qubit

A qubit starts in $|0\rangle$. `Statevector` shows its amplitudes
without measuring, which only a simulator can do.

In [ ]:
qc = QuantumCircuit(1)
state = Statevector(qc)
print(state)
plot_bloch_multivector(state)

## 1.2 The X gate (quantum NOT)

X flips $|0\rangle$ to $|1\rangle$: the arrow moves from the north pole
to the south pole.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
plot_bloch_multivector(Statevector(qc))

## 1.3 The H gate (superposition)

Hadamard takes $|0\rangle$ to
$\frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, a point on the equator.
Both amplitudes are $1/\sqrt{2} \approx 0.707$, so each outcome has
probability $0.707^2 = 0.5$.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
state = Statevector(qc)
print(state)
print("Probabilities:", state.probabilities_dict())
plot_bloch_multivector(state)

## 1.4 Measurement

Measuring collapses the qubit to 0 or 1. Repeating the circuit many times
("shots") reveals the probabilities.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.measure_all()
display(qc.draw("mpl"))
counts = run(qc, shots=1000)
print(counts)
plot_histogram(counts)

Try changing `shots` to 10, then 10000. With few shots the split is
uneven; with many it approaches 50/50.

## Exercise 1.1: Always 1

Build a circuit that measures `1` every time.

In [ ]:
qc = QuantumCircuit(1)
qc.x(0)
qc.measure_all()
counts = run(qc)
print(counts)
assert counts == {"1": 1000}

## Exercise 1.2: Interference

Apply H twice, then measure. Predict the result before you run it.
Is it 50/50?

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
qc.measure_all()
counts = run(qc)
print(counts)
assert counts == {"0": 1000}

The answer is always 0. The two paths to `1` have opposite signs and
cancel, while the paths to `0` add up. This *interference* is what
quantum algorithms use to boost right answers and cancel wrong ones.
A classical coin flipped twice would still be 50/50.

## Exercise 1.3: A biased quantum coin

`qc.ry(theta, 0)` rotates the qubit by angle `theta` about the Y axis.
The probability of measuring 0 is $\cos^2(\theta/2)$.

Find `theta` so the qubit measures 0 about 75% of the time.

In [ ]:
theta = 2 * np.arccos(np.sqrt(0.75))  # = pi/3

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()
counts = run(qc, shots=10000)
print(counts)
assert abs(counts.get("0", 0) / 10000 - 0.75) < 0.03
plot_histogram(counts)

## Summary

- A qubit's state is a pair of amplitudes; measurement probabilities are
  their squares.
- H creates an equal superposition. X flips the qubit.
- Amplitudes can cancel (interference), which has no classical analogue.